In [0]:
!pip install openpyxl pydub moviepy ffmpeg-python

In [0]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import re
from pydub import AudioSegment
from pydub.silence import detect_silence
from pydub.playback import play
from IPython.display import Audio
from moviepy import VideoFileClip
import ffmpeg
import tempfile
import json
import io
from math import floor, ceil
from collections import Counter
from tqdm import tqdm
import ast
import glob

In [0]:
AUDIO_FILES = glob.glob("")
TRANSCRIPTS = glob.glob("")
AUDIO_FILES = sorted(AUDIO_FILES, key=lambda x: x.split('/')[-1])  # Sort by file name
TRANSCRIPTS = sorted(TRANSCRIPTS, key=lambda x: x.split('/')[-1])
print(len(TRANSCRIPTS), len(AUDIO_FILES))

In [0]:
replacements = {
    "Reponsiveness": "Responsiveness",
    "Sympathethic": "Sympathetic",
    "Interest attentiveness": "Interest/Attentiveness",
    "Interest Attentiveness": "Interest/Attentiveness",
    "Interest/attentiveness": "Interest/Attentiveness",
    "Attentivness": "Attentiveness",
    "Nerviousness": "Nervousness",
    "Warmth": "Warm",
    "Int.": "Interactivity",
    "\n": "",
    "Responsiveness/Awareness": "Responsiveness/Engagement",
    "it": "None"
}

def visit_name(name):
    return name.replace("_trimmed.xlsx", "").split("_room_view")[0].split("/")[-1]

def visit_length_format(visit_length_seconds):
    return f"[{int(visit_length_seconds // 60):02}:{int(visit_length_seconds % 60):02}]" # Convert seconds to '[mm:ss]' format

def get_transcripts(transcripts):
    # Processing transcripts
    dfs = []
    for e, transcript in tqdm(enumerate(transcripts), total=len(transcripts)):
        df0 = pd.read_excel(transcript)
        df0["line_number"] = df0.index + 1
        df0["visit"] = visit_name(transcript)
        df0["Affect"] = df0["Affect"].fillna("None")
        visit_length_seconds = df0["visit_length_seconds"].iloc[0]
        df0.loc[len(df0), "Timestamp"] = visit_length_format(visit_length_seconds)
        df0["Affect"] = df0["Affect"].replace(replacements, regex=True)
        dfs.append(df0)

    return dfs

def get_silence_timestamps(audio_files):
    # Processing silence timestamps
    silences = []
    for audio_file in tqdm(audio_files):
        audio_segment = AudioSegment.from_file(audio_file, format="wav")

        silence = detect_silence(
            audio_segment,
            min_silence_len=300, # ms, 0.3 seconds
            silence_thresh=-45 # default silence thresh
        )

        silence = [[start / 1000, end / 1000] for start, end in silence]
        silences.append(silence)

    return silences

In [0]:
transcript_dfs = get_transcripts(TRANSCRIPTS)
# silences = get_silence_timestamps(AUDIO_FILES) # run once to get silences ~ 40 min run time
silences = pd.read_csv("silences_03_duration.csv")
silences = [ast.literal_eval(s) for s in silences["silences"]]

In [0]:
def convert_to_seconds(df):
    df["Timestamp"] = df["Timestamp"].astype(str).str.replace(r"[\[\].]", "", regex=True)
    df["Timestamp"] = df["Timestamp"].apply(lambda x: sum(int(i) * j for i, j in zip(x.split(":"), [60, 1]))) # Convert the 'mm:ss' to total seconds
    return df

def convert_to_format(df):
    df["Timestamp"] = df["Timestamp"].apply(lambda x: f"[{int(x // 60):02}:{int(x % 60):02}]") # Convert back to '[mm:ss]' format
    return df

def silence_timestamps_and_durations(df, silences):
    """
    Processes a DataFrame to insert silence timestamps and their durations.

    This function iterates through a DataFrame containing timestamps, calculates silence durations
    based on provided silence intervals, and assigns them to appropriate rows. Silences are inserted
    either to the current speaker's row or the next speaker's row based on the end silence timestamp.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing a 'Timestamp' column in a format convertible to seconds.
    silences : list of tuples
        List of tuples where each tuple contains (start, end) timestamps of silence intervals.

    Returns:
    --------
    pandas.DataFrame
        Modified DataFrame with two additional columns:
        - 'start_end_silence_timestamps': String representation of silence intervals.
        - 'total_silence_duration': Duration of silence in seconds, rounded to three decimal places.

    Notes:
    ------
    - Timestamps in the DataFrame are converted to seconds for processing and converted back to their
      original format before returning.
    - Each silence interval is processed only once.
    - Silence duration is calculated as the difference between end and start timestamps.
    - If the end silence timestamp is greater than or equal to the next timestamp, the silence is
      assigned to the next row; otherwise, it is assigned to the current row.
    """
    df = convert_to_seconds(df)

    appended_silences = set()
    silence_durations, locs = [], []
    for idx in range(len(df) - 1):
        for silence in silences:
            silence_tuple = tuple(silence)

            if silence_tuple in appended_silences or silence_tuple[0] >= df["Timestamp"].iloc[idx + 1]:
                continue
        
            # 1. iterate through timestamps
            # 2. calculate the silence duration where the start silence timestamp > current timestamp
            # 3. if the end silence timestamp >= the next timestamp: insert the silence to the next speaker's row
            # 4. if the end silence timestamp < the next timestamp: insert the silence to the current speaker's row

            silence_duration = round(silence_tuple[1] - silence_tuple[0], 3)
            appended_silences.add(silence_tuple)
            silence_durations.append(silence_duration)

            if silence_tuple[1] >= df["Timestamp"].iloc[idx + 1]:
                locs.append(idx + 1)
            else:
                locs.append(idx)

    df = convert_to_format(df)

    df["start_end_silence_timestamps"] = ""
    df["total_silence_duration"] = 0
    for idx, loc in enumerate(locs):
        df.loc[loc, "start_end_silence_timestamps"] = str(silences[idx])
        df.loc[loc, "total_silence_duration"] = silence_durations[idx]

    return df

In [0]:
transcript_dfs = [silence_timestamps_and_durations(transcript_dfs[i], silences[i]) for i in tqdm(range(len(TRANSCRIPTS)))]

In [0]:
def sort_timestamps(df):
    df = df.copy()
    df["Timestamp"] = df["Timestamp"].astype(str).str.replace(r"[\[\].]", "", regex=True)
    df["Timestamp"] = df["Timestamp"].apply(lambda x: sum(int(i) * j for i, j in zip(x.split(":"), [60, 1]))) # Convert the 'mm:ss' to total seconds
    df = df.sort_values(by="Timestamp", ignore_index=True)
    df["Timestamp"] = df["Timestamp"].apply(lambda x: f"[{int(x // 60):02}:{int(x % 60):02}]") # Convert back to 'mm:ss' format
    return df


def process_time(row):
    return row["Timestamp"].strip().replace("[", "").replace("]", "").replace(".", "")


def colon_count(count):
    if count == 2: return "%H:%M:%S"
    elif count == 1: return "%M:%S"
    else: raise ValueError("Unexpected timestamp format")


def calculate_duration(df, itr):
    current_ts_str = str(process_time(df.iloc[itr-1]))
    next_ts_str = str(process_time(df.iloc[itr]))
    current_colon_count, next_colon_count = current_ts_str.count(":"), next_ts_str.count(":")
    current_time_format = colon_count(current_colon_count)
    next_time_format = colon_count(next_colon_count)
    current_ts = datetime.strptime(process_time(df.iloc[itr-1]), current_time_format)
    next_ts = datetime.strptime(process_time(df.iloc[itr]), next_time_format)
    duration = (next_ts - current_ts).seconds
    return duration


def display_vals(pct, allvals):
    absolute = int(pct/100.*sum(allvals))
    return "{:.1f}%\n({:d}s)".format(pct, absolute)


def check_proficiency(proficiency, target, speaker_type, current_speaker, duration):
    if (pd.notna(proficiency) and 
        isinstance(proficiency, str) and 
        target in proficiency.lower() and 
        speaker_type == current_speaker):
        return duration
    return 0


def speaker_durations(df, plot_dist=True):
    durations = {
        'Provider': [],
        'Patient': [],
        'Other': []
    }

    proficiencies = {
        'Provider_cell_phone_use': [],
        'Provider_computer_use': [],
        'Patient_cell_phone_use': [],
        'Patient_computer_use': []
    }

    affects = {
        "Provider_Sympathetic/Empathetic": [], "Patient_Sympathetic/Empathetic": [], "Other_Sympathetic/Empathetic": [],
        "Provider_Dominance/Assertiveness": [], "Patient_Dominance/Assertiveness": [], "Other_Dominance/Assertiveness": [],
        "Provider_Respectfulness": [], "Patient_Respectfulness": [], "Other_Respectfulness": [],
        "Provider_None": [], "Patient_None": [], "Other_None": [],
        "Provider_Friendliness/Warm": [], "Patient_Friendliness/Warm": [], "Other_Friendliness/Warm": [],
        "Provider_Interactivity": [], "Patient_Interactivity": [], "Other_Interactivity": [],
        "Provider_Interest/Attentiveness": [], "Patient_Interest/Attentiveness": [], "Other_Interest/Attentiveness": [],
        "Provider_Responsiveness/Engagement": [], "Patient_Responsiveness/Engagement": [], "Other_Responsiveness/Engagement": [],
        "Provider_Emotional Distress/Upset": [], "Patient_Emotional Distress/Upset": [], "Other_Emotional Distress/Upset": [],
        "Provider_Depression/Sadness": [], "Patient_Depression/Sadness": [], "Other_Depression/Sadness": [],
        "Provider_Anger/Irritation": [], "Patient_Anger/Irritation": [], "Other_Anger/Irritation": []
    }

    laughs = {
        "Provider_total": 0, "Patient_total": 0, "Other_total": 0
    }

    point_5_count = 0
    for i in range(1, len(df)):
        duration = calculate_duration(df, i)
        speaking_duration = duration
        speaking_duration -= df.iloc[i-1]["total_silence_duration"]
        speaker = df.iloc[i-1]["Speaker"]
        
        if duration == 0: 
            duration = 0.5
            speaking_duration = 0.5
            point_5_count +=1

        durations['Provider'].append(speaking_duration if speaker == 'Doctor' else 0)
        durations['Patient'].append(speaking_duration if speaker == 'Patient' else 0)
        durations['Other'].append(speaking_duration if speaker not in ['Doctor', 'Patient'] else 0)


        proficiencies['Provider_cell_phone_use'].append(
            check_proficiency(df.iloc[i-1]["Proficiency"], "phone", "Doctor", speaker, duration)
        )
        proficiencies['Provider_computer_use'].append(
            check_proficiency(df.iloc[i-1]["Proficiency"], "computer", "Doctor", speaker, duration)
        )
        proficiencies['Patient_cell_phone_use'].append(
            check_proficiency(df.iloc[i-1]["Proficiency"], "phone", "Patient", speaker, duration)
        )
        proficiencies['Patient_computer_use'].append(
            check_proficiency(df.iloc[i-1]["Proficiency"], "computer", "Patient", speaker, duration)
        )


        affect_combo = df.iloc[i-1]["Affect"]
        all_affects = [affect.strip() for affect in affect_combo.split('; ')]
        
        base_affects = ["Sympathetic/Empathetic", "Dominance/Assertiveness", "Respectfulness",
                        "None", "Friendliness/Warm", "Interactivity", "Interest/Attentiveness",
                        "Responsiveness/Engagement", "Emotional Distress/Upset", "Depression/Sadness",
                        "Anger/Irritation"]
        
        def update_affects(prefix, base_affect):
            roles = {"Provider": 0, "Patient": 0, "Other": 0}
            if base_affect in all_affects:
                roles[prefix] = duration
            for role, value in roles.items():
                affects[f"{role}_{base_affect}"].append(value)

        for base_affect in base_affects:
            if speaker == "Doctor":
                update_affects("Provider", base_affect)
            elif speaker == "Patient":
                update_affects("Patient", base_affect)
            else:
                update_affects("Other", base_affect)
    

        if isinstance(df["Transcript"].iloc[i], str) and "[laughs]" in df["Transcript"].iloc[i]:
            if speaker == "Doctor":
                laughs["Provider_total"] += 1
            elif speaker == "Patient":
                laughs["Patient_total"] += 1
            elif speaker not in ["Patient", "Provider"]:
                laughs["Other_total"] += 1

    # print("total sum", sum(durations["Provider"] + durations["Patient"] + durations["Other"]))
    # print(sum(proficiencies["Provider_cell_phone_use"]), sum(proficiencies["Provider_computer_use"]), sum(proficiencies["Patient_cell_phone_use"]), sum(proficiencies["Patient_computer_use"]))
    # print("point 5 count", point_5_count)

    if plot_dist:
        provider_sum, patient_sum, other_sum = sum(durations["Provider"]), sum(durations["Patient"]), sum(durations["Other"])
        labels = ['Provider', 'Patient', 'Silence']
        sizes = [provider_sum, patient_sum, df.total_silence_duration.sum()]
        colors = ['lightcoral', 'yellowgreen', 'gray']
        if other_sum > 0:
            labels.append("Other")
            sizes.append(other_sum)
            colors.append("gold")

        plt.figure(figsize=(8, 8))
        plt.pie(sizes, labels=labels, colors=colors, autopct=lambda pct: display_vals(pct, sizes), startangle=240)
        plt.axis('equal')
        plt.title(df.iloc[0]["visit"])
        plt.savefig(f'{df.iloc[0]["visit"]}.png')
        plt.show()

    return pd.DataFrame({
        "provider_speaking_durations": durations["Provider"],
        "patient_speaking_durations": durations["Patient"],
        "other_speaking_durations": durations["Other"],
        "provider_silence_durations": [oth_dur + pt_dur if oth_dur + pt_dur > 0.5 else 0 for oth_dur, pt_dur in zip(durations["Other"], durations["Patient"])],
        "patient_silence_durations": [oth_dur + pr_dur if oth_dur + pr_dur > 0.5 else 0 for oth_dur, pr_dur in zip(durations["Other"], durations["Provider"])],
        "other_silence_durations": [pt_dur + pr_dur if pt_dur + pr_dur > 0.5 else 0 for pt_dur, pr_dur in zip(durations["Patient"], durations["Provider"])],
        "total_silence_duration": df.iloc[:-1]["total_silence_duration"],
        "provider_cell_phone_use": proficiencies["Provider_cell_phone_use"],
        "provider_computer_use": proficiencies["Provider_computer_use"],
        "patient_cell_phone_use": proficiencies["Patient_cell_phone_use"],
        "patient_computer_use": proficiencies["Patient_computer_use"],
        "provider_sympathetic_empathetic": affects["Provider_Sympathetic/Empathetic"],
        "patient_sympathetic_empathetic": affects["Patient_Sympathetic/Empathetic"],
        "other_sympathetic_empathetic": affects["Other_Sympathetic/Empathetic"],
        "provider_dominance_assertiveness": affects["Provider_Dominance/Assertiveness"],
        "patient_dominance_assertiveness": affects["Patient_Dominance/Assertiveness"],
        "other_dominance_assertiveness": affects["Other_Dominance/Assertiveness"],
        "provider_respectfulness": affects["Provider_Respectfulness"],
        "patient_respectfulness": affects["Patient_Respectfulness"],
        "other_respectfulness": affects["Other_Respectfulness"],
        "provider_none": affects["Provider_None"],
        "patient_none": affects["Patient_None"],
        "other_none": affects["Other_None"],
        "provider_friendliness_warm": affects["Provider_Friendliness/Warm"],
        "patient_friendliness_warm": affects["Patient_Friendliness/Warm"],
        "other_friendliness_warm": affects["Other_Friendliness/Warm"],
        "provider_interactivity": affects["Provider_Interactivity"],
        "patient_interactivity": affects["Patient_Interactivity"],
        "other_interactivity": affects["Other_Interactivity"],
        "provider_interest_attentiveness": affects["Provider_Interest/Attentiveness"],
        "patient_interest_attentiveness": affects["Patient_Interest/Attentiveness"],
        "other_interest_attentiveness": affects["Other_Interest/Attentiveness"],
        "provider_responsiveness_engagement": affects["Provider_Responsiveness/Engagement"],
        "patient_responsiveness_engagement": affects["Patient_Responsiveness/Engagement"],
        "other_responsiveness_engagement": affects["Other_Responsiveness/Engagement"],
        "provider_emotional_distress_upset": affects["Provider_Emotional Distress/Upset"],
        "patient_emotional_distress_upset": affects["Patient_Emotional Distress/Upset"],
        "other_emotional_distress_upset": affects["Other_Emotional Distress/Upset"],
        "provider_depression_sadness": affects["Provider_Depression/Sadness"],
        "patient_depression_sadness": affects["Patient_Depression/Sadness"],
        "other_depression_sadness": affects["Other_Depression/Sadness"],
        "provider_anger_irritation": affects["Provider_Anger/Irritation"],
        "patient_anger_irritation": affects["Patient_Anger/Irritation"],
        "other_anger_irritation": affects["Other_Anger/Irritation"],
        "provider_laugh_total": laughs["Provider_total"],
        "patient_laugh_total": laughs["Patient_total"],
        "other_laugh_total": laughs["Other_total"],
        "visit_length_seconds": df.iloc[0]["visit_length_seconds"],
        "visit": df.iloc[0]["visit"],
    })

In [0]:
result_df = pd.DataFrame()
for transcript_df in tqdm(transcript_dfs):
    temp_df = speaker_durations(transcript_df, plot_dist=True)
    result_df = pd.concat([result_df, temp_df], ignore_index=True)